In [2]:
import pandas as pd

rides = pd.read_parquet('../data/transformed/validated_rides_2024_06.parquet')
rides.head(10)

,pickup_datetime,pickup_location_id
0,2024-06-01 00:03:46,138
1,2024-06-01 00:55:22,138
2,2024-06-01 00:23:53,166
3,2024-06-01 00:32:24,148
4,2024-06-01 00:51:38,148
5,2024-06-01 00:26:13,48
6,2024-06-01 00:01:04,132
7,2024-06-01 00:43:55,140
9,2024-06-01 00:00:09,142
10,2024-06-01 00:16:07,48


In [3]:
rides['pickup_datetime'].dtype

dtype('<M8[us]')

In [4]:
rides['pickup_hour'] = rides['pickup_datetime'].dt.floor('h')
rides

,pickup_datetime,pickup_location_id,pickup_hour
0,2024-06-01 00:03:46,138,2024-06-01 00:00:00
1,2024-06-01 00:55:22,138,2024-06-01 00:00:00
2,2024-06-01 00:23:53,166,2024-06-01 00:00:00
3,2024-06-01 00:32:24,148,2024-06-01 00:00:00
4,2024-06-01 00:51:38,148,2024-06-01 00:00:00
...,...,...,...
3539188,2024-06-30 23:07:36,255,2024-06-30 23:00:00
3539189,2024-06-30 23:46:07,68,2024-06-30 23:00:00
3539190,2024-06-30 23:18:50,41,2024-06-30 23:00:00
3539191,2024-06-30 23:33:36,158,2024-06-30 23:00:00


In [5]:
agg_rides = rides.groupby(['pickup_hour', 'pickup_location_id']).size().reset_index()
agg_rides.rename(columns={0: 'rides'}, inplace=True)
agg_rides

,pickup_hour,pickup_location_id,rides
0,2024-06-01 00:00:00,4,34
1,2024-06-01 00:00:00,7,9
2,2024-06-01 00:00:00,13,10
3,2024-06-01 00:00:00,17,4
4,2024-06-01 00:00:00,20,1
...,...,...,...
89804,2024-06-30 23:00:00,261,15
89805,2024-06-30 23:00:00,262,8
89806,2024-06-30 23:00:00,263,30
89807,2024-06-30 23:00:00,264,5


In [6]:
from tqdm import tqdm

def add_missing_slots(agg_rides: pd.DataFrame) -> pd.DataFrame:
    
    location_ids = agg_rides['pickup_location_id'].unique()
    full_range = pd.date_range(
        agg_rides['pickup_hour'].min(), agg_rides['pickup_hour'].max(), freq='h'
    )
    output = pd.DataFrame()
    for location_id in tqdm(location_ids):
        
        # keep only rides for this location ids
        agg_rides_i = agg_rides.loc[agg_rides.pickup_location_id == location_id, ['pickup_hour', 'rides']]
        
        # quick way to add missing hours with 0 rides
        agg_rides_i.set_index('pickup_hour', inplace=True)
        agg_rides_i.index = pd.DatetimeIndex(agg_rides_i.index)
        agg_rides_i = agg_rides_i.reindex(full_range, fill_value=0)
        
        # add back location id column
        agg_rides_i['pickup_location_id'] = location_id
        output = pd.concat([output, agg_rides_i])
        
    output = output.reset_index().rename(columns={'index': 'pickup_hour'})
    return output

In [7]:
agg_rides_all_slots = add_missing_slots(agg_rides)
agg_rides_all_slots

100%|██████████| 260/260 [00:00<00:00, 765.06it/s]


,pickup_hour,rides,pickup_location_id
0,2024-06-01 00:00:00,34,4
1,2024-06-01 01:00:00,63,4
2,2024-06-01 02:00:00,56,4
3,2024-06-01 03:00:00,30,4
4,2024-06-01 04:00:00,13,4
...,...,...,...
187195,2024-06-30 19:00:00,0,176
187196,2024-06-30 20:00:00,0,176
187197,2024-06-30 21:00:00,0,176
187198,2024-06-30 22:00:00,0,176


In [8]:
from typing import Optional, List
import plotly.express as px
import pandas as pd

def plot_rides(
    rides: pd.DataFrame,
    locations: Optional[List[int]] = None
) -> None:
    
    """
    Plot time-series data
    """

    rides_to_plot = rides[rides.pickup_location_id.isin(locations)] if locations else rides

    fig = px.line(
        rides_to_plot,
        x='pickup_hour',
        y='rides',
        color='pickup_location_id',
        title='Rides per hour by pickup location'
    )

    fig.show()


In [9]:
plot_rides(agg_rides_all_slots, locations=[43])

In [10]:
agg_rides_all_slots.to_parquet('../data/transformed/transformed_ts_2024_06.parquet')